In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
LONG_COMP = ['VCVS', 'CCVS', 'VCCS', 'CCCS', 'ISC']
BASIC = ['R', 'V', 'I', 'C', 'L']
NLIN = ['D']

def get_type(name):
    name = name.upper()
    for prefix in LONG_COMP:
        if name.startswith(prefix):
            return prefix
    return name[0]

def val_comp_format(name, comp_type, comp):
    #Supported component validation
    if comp_type not in BASIC and comp_type not in LONG_COMP and comp_type not in NLIN:
        raise ValueError('Unsupported component type', name)

    #Component format validation
    if comp_type == 'ISC':
        if len(comp) != 3:
            raise ValueError('Expected ISCname node+ node-', name)
        if comp[1] == comp[2]:
            raise ValueError('Sensing-short terminals must differ', name)
    elif comp_type == 'D':
        if len(comp) != 3:
            raise ValueError('Expected name anode cathode', name)
    elif comp_type in ('R', 'C', 'L'):
        if len(comp) != 4:
            raise ValueError('Expected name node+ node- value', name)
    elif comp_type == 'I':
        if len(comp) < 5 or len(comp) > 6:
            raise ValueError('Expected name node+ node- TYPE value', name)
        source_type = comp[3]
        
        if source_type == 'DC':
            if len(comp) != 5:
                raise ValueError('Expected name node+ node- DC value', name)
        elif source_type in ('SIN', 'COS'):
            if len(comp) != 6:
                raise ValueError('Expected name node+ node- TYPE amp freq', name)
        else:
            raise ValueError('Source type must be DC, SIN, or COS')
    elif comp_type == 'V':
        if len(comp) < 5 or len(comp) > 6:
            raise ValueError('Expected name node+ node- TYPE value', name)
        source_type = comp[3]
        
        if source_type == 'DC':
            if len(comp) != 5:
                raise ValueError('Expected name node+ node- DC value', name)
        elif source_type in ('SIN', 'COS'):
            if len(comp) != 6:
                raise ValueError('Expected name node+ node- TYPE amp freq', name)
        else:
            raise ValueError('Source type must be DC, SIN, or COS')
    elif comp_type in ('VCVS', 'VCCS'):
        if len(comp) != 6:
            raise ValueError('Expected name out+ out- control+ control- gain', name)
    elif comp_type in ('CCCS', 'CCVS'):
        if len(comp) != 5:
            raise ValueError('Expected name out+ out- controlling_component gain', name)

def val_dir_format(name, parts):
    if name == '.DC':
        if len(parts) > 1:
            raise ValueError('Directive .DC must be in form .DC', parts)
    elif name == '.AC':
        if len(parts) > 1:
            raise ValueError('Directive .AC must be in form .AC', parts)
    elif name == '.TRAN':
        if len(parts) < 2 or len(parts) > 3:
            raise ValueError('Directive .TRAN must be in form .TRAN duration stepsize or .TRAN duration', parts)
    else:
        raise ValueError('Unsupported directive, please enter either .TRAN, .AC, or .DC')
        
def read_netlist(file):
    comps, dirs = [], []
    seen_names = set()
    found_end = False
    with open(file) as f:
        title = f.readline().rstrip('\n') 
        for line in f:
            line = line.strip().upper()

            #empty space or comments
            if not line or line.startswith('*'):
                continue

            #.end
            if found_end:
                raise ValueError('Content after .END', line)
            if line == '.END':
                found_end = True
                continue


            parts = line.split()
            name = parts[0]
            
            #directive control
            if name.startswith('.'):
                val_dir_format(name, parts)
                dirs.append(line)
                continue
            
            comp_type = get_type(name)
            val_comp_format(name, comp_type, parts)


            if name in seen_names:
                raise ValueError('Duplicate component', name)
            seen_names.add(name)
            comps.append(parts)

    if not found_end:
        raise ValueError('Missing .END')
    if len(dirs) != 1:
        raise ValueError('Expected exactly one active directive')
    if not comps:
        raise ValueError('Cannot calculate circuit with no components')
    return title, comps, dirs[0]

In [ ]:
def gen_matrices(comps):
    node_index, cur_index = {}, {}
    size = 0
    for comp in comps:
        name = comp[0]
        kind = get_type(name)
        nodes = comp[1:3]
        if kind in ('VCVS', 'VCCS'):
            nodes = nodes + comp[3:5]
        for node in nodes:
            if node not in node_index:
                if node != '0':
                    node_index[node] = size
                    size += 1 
                else:
                    node_index[node] = None
        if kind in ('V', 'VCVS', 'CCVS', 'L', 'ISC'):
            cur_index['I' + name] = size
            size += 1
    
    G = np.zeros((size, size), dtype=float)
    C = np.zeros((size, size), dtype=float)
    b = np.zeros(size, dtype=float)
    return G, C, b, node_index, cur_index

In [ ]:
def numeric_value(text, name):
    try:
        return float(text)
    except ValueError:
        raise ValueError(f'{name}: expected a numerical value, got {text!r}') from None

def fill_matrices(comps, G, C, node_index, cur_index):
    def add_G(row, col, value):
        if row is not None and col is not None:
            G[row, col] += value

    def add_C(row, col, value):
        if row is not None and col is not None:
            C[row, col] += value
            
    def stamp_voltage_branch(i, j, current):
        add_G(j, current, -1)
        add_G(current, j, -1)
        add_G(i, current, 1)
        add_G(current, i, 1)

    G.fill(0.0)
    C.fill(0.0)
    diodes = []
    insources = []
    for comp in comps:
        name = comp[0]
        comp_type = get_type(name)

        i = node_index[comp[1]]
        j = node_index[comp[2]]

        if comp_type == 'R':
            r = numeric_value(comp[3], name)

            if not np.isfinite(r) or r <= 0:
                raise ValueError('Resistance must be finite and positive')

            g = 1.0 / r

            add_G(i, i, g)
            add_G(i, j, -g)
            add_G(j, i, -g)
            add_G(j, j, g)

        elif comp_type == 'ISC':
            ie = cur_index['I' + name]
            stamp_voltage_branch(i, j, ie)
            
        elif comp_type == 'I':
            insources.append(comp)
            continue
        elif comp_type == 'V':
            ie = cur_index['I' + name]
            stamp_voltage_branch(i, j, ie)
            insources.append(comp)
            
        elif comp_type == 'C':
            F = numeric_value(comp[3], name)

            if not np.isfinite(F) or F < 0:
                raise ValueError('Capacitance must be finite and non-negative')

            add_C(i, i, F)
            add_C(i, j, -F)
            add_C(j, i, -F)
            add_C(j, j, F)

        elif comp_type == 'L':
            H = numeric_value(comp[3], name)

            if not np.isfinite(H) or H < 0:
                raise ValueError('Inductance must be finite and non-negative')

            il = cur_index['I' + name]

            stamp_voltage_branch(i, j, il)
            add_C(il, il, -H)

        elif comp_type == 'VCCS':
            cplus = node_index[comp[3]]
            cminus = node_index[comp[4]]
            gm = numeric_value(comp[5], name)

            if not np.isfinite(gm):
                raise ValueError('VCCS gain must be finite')

            add_G(i, cplus, gm)
            add_G(i, cminus, -gm)
            add_G(j, cplus, -gm)
            add_G(j, cminus, gm)

        elif comp_type == 'VCVS':
            cplus = node_index[comp[3]]
            cminus = node_index[comp[4]]
            A = numeric_value(comp[5], name)
            ie = cur_index['I' + name]

            if not np.isfinite(A):
                raise ValueError('VCVS gain must be finite')

            stamp_voltage_branch(i, j, ie)
            add_G(ie, cplus, -A)
            add_G(ie, cminus, A)

        elif comp_type == 'CCCS':
            isense = cur_index['I' + comp[3]]
            gain = numeric_value(comp[4], name)

            if not np.isfinite(gain):
                raise ValueError('CCCS gain must be finite')

            add_G(i, isense, gain)
            add_G(j, isense, -gain)

        elif comp_type == 'CCVS':
            ie = cur_index['I' + name]
            isense = cur_index['I' + comp[3]]
            gain = numeric_value(comp[4], name)

            if not np.isfinite(gain):
                raise ValueError('CCVS gain must be finite')

            stamp_voltage_branch(i, j, ie)
            add_G(ie, isense, -gain)

        elif comp_type == 'D':
            diodes.append(comp)
            
    return G, C, diodes, insources
    
def fill_diodes(diodes, x, node_index, Is=1e-15, Vt=0.025):
    size = len(x)
    f = np.zeros(size, dtype=float)
    J = np.zeros((size, size), dtype=float)

    with np.errstate(over='raise', invalid='raise', divide='raise'):
        for diode in diodes:
            i = node_index[diode[1]]
            j = node_index[diode[2]]

            vi = 0.0 if i is None else x[i]
            vj = 0.0 if j is None else x[j]
            vd = vi - vj

            exp_term = np.exp(vd / Vt)
            Id = Is * np.expm1(vd / Vt)
            gd = (Is / Vt) * exp_term

            if i is not None:
                f[i] += Id
                J[i, i] += gd

            if j is not None:
                f[j] -= Id
                J[j, j] += gd

            if i is not None and j is not None:
                J[i, j] -= gd
                J[j, i] -= gd
    return f, J

def fill_sources(insources, b, node_index, cur_index, t=0.0):
    b_new = np.zeros_like(b, dtype=float)

    def add_b(row, value):
        if row is not None:
            b_new[row] += value

    for source in insources:
        name = source[0]
        source_type = get_type(name)
        source_out = source[3]
        amplitude = numeric_value(source[4], name)

        if source_out == 'DC':
            value = amplitude

        else:
            frequency = numeric_value(source[5], name)
            angle = 2 * np.pi * frequency * t
            if source_out == 'SIN':
                value = amplitude * np.sin(angle)
            else:
                value = amplitude * np.cos(angle)

        if source_type == 'V':
            ie = cur_index['I' + name]
            add_b(ie, value)

        else:
            i = node_index[source[1]]
            j = node_index[source[2]]

            add_b(i, -value)
            add_b(j, value)

    return b_new

In [ ]:
def lin_dc_analysis(G, b):
    try: 
        x = np.linalg.solve(G, b)
    except np.linalg.LinAlgError as e:
        raise ValueError('No unique DC Solution') from e
    return x

def nlin_dc_analysis(G, b, diodes, node_index, estep_max=1.0, cont_steps=50, cont=False):
    x0 = np.zeros(len(b), dtype=float)
    
    if init_guess:
        for diode in diodes:
            i = node_index[diode[1]]
            if i is not None:
                x0[i] = 0.7

    try:
        x, estep = newton_raphson(G, b, x0, diodes, node_index, alpha=1.0)
        if estep <= estep_max:
            return x
            
        raise RuntimeError(f'Newton-Raphson relative step {estep:.6g} 'f'exceeds the limit {estep_max:.6g}')
    
        #print('Relative-step check failed; trying continuation.')
    except (RuntimeError, FloatingPointError,np.linalg.LinAlgError) as e:
        if not cont:
            raise
            
        print(f'Normal Newton-Raphson failed: {e}')
        print('Trying continuation.')

    if not isinstance(cont_steps, int) or cont_steps <= 0:
        raise ValueError('cont_steps must be a positive integer')
    
    #Newton-Raphson with Continuation
    x = np.zeros(len(b), dtype=float)
    for step in range(1, cont_steps + 1):
        alpha = step / cont_steps
    
        try:
            x, _ = newton_raphson(G, b, x, diodes, node_index, alpha)
        except (RuntimeError,FloatingPointError,np.linalg.LinAlgError) as e:
            raise RuntimeError(f'Continuation failed at alpha={alpha}') from e
    return x

def newton_raphson(G, b, x, diodes, node_index, alpha=1.0, max_iters=100, thres=1e-6, denom_floor=1e-12):
    x = np.array(x, dtype=float, copy=True)
    estep = 100
    with np.errstate(over='raise', invalid='raise', divide='raise'):
        for iteration in range(max_iters):
            f, Jnl = fill_diodes(diodes, x, node_index)

            F = G @ x + f - alpha * b
            J = G + Jnl

            delta_x = np.linalg.solve(J, -F)
            x = x + delta_x

            if not np.all(np.isfinite(x)):
                raise FloatingPointError('Nonfinite Newton iterate')

            if np.linalg.norm(delta_x) <= thres:
                estep = (np.linalg.norm(delta_x)/ max(np.linalg.norm(x), denom_floor))
                return x, estep
                
    raise RuntimeError(f'Newton-Raphson did not converge within {max_iters} iterations')

def parse_time(text):
    units = {'MS': 1e-3, 'US': 1e-6, 'NS': 1e-9, 'S': 1.0}

    for suffix, scale in units.items():
        if text.endswith(suffix):
            value = float(text[:-len(suffix)]) * scale
            break
    else:
        value = float(text)  # No suffix means seconds.

    if not np.isfinite(value) or value <= 0:
        raise ValueError('Time must be finite and positive')
    return value

def tran_analysis(G, C, b, diodes, node_index, cur_index, insources, parts):
    #Time control
    dur = parse_time(parts[1])
    stepsize = (parse_time(parts[2]) if len(parts) == 3 else dur / 100)
    steps = int(np.ceil(dur / stepsize)) 

    xtnow = np.zeros(len(b), dtype=float)
    full_x = []
    
    for i in range(0, steps):
        #setup
        tnext = (i + 1) * stepsize
        btnext = fill_sources(insources, b, node_index, cur_index, tnext)
        ch = C / stepsize
        left = G + ch
        right = btnext + ch @ xtnow

        #solve for next x
        if diodes:
            xtnext, _ = newton_raphson(left, right, xtnow, diodes, node_index)
        else:
            xtnext = np.linalg.solve(left, right)

        #advance t
        full_x.append(xtnext)
        xtnow = xtnext
        
    return xtnow, full_x, stepsize

In [ ]:
def printx(x, node_index, cur_index):
    for key in node_index:
        if key == '0':
            continue
        index = node_index[key]
        val = x[index]
        print(key + ":" + str(val) + ' V')
    for key in cur_index:
        index = cur_index[key]
        val = x[index]
        print(key + ":" + str(val) + ' A')

def plot(full_x, node_index, stepsize):
    full_x = np.asarray(full_x)
    times = np.arange(len(full_x)) * stepsize

    plt.figure()
    plt.plot(times, full_x[:, node_index['N1']], label='Voltage at N1')
    plt.plot(times, full_x[:, node_index['N2']], label='Voltage at N2')

    plt.xlabel('Time (s)')
    plt.ylabel('Voltage (V)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
def run_netlist(file='data.txt', show_matrices=True):
    title, comps, act_dir = read_netlist(file)
    G, C, b, node_index, cur_index = gen_matrices(comps)
    G, C, diodes, insources = fill_matrices(comps, G, C, node_index, cur_index)
    if show_matrices:
        print(node_index)
        print(cur_index)
        print(G)
        print(C)

    print(insources)
    #directive handling
    parts = act_dir.split()
    dir_type = parts[0]
    
    if dir_type == '.DC':
        b = fill_sources(insources, b, node_index, cur_index, t=0.0)
        if diodes:
            print('Running nonlinear DC analysis')
            x = nlin_dc_analysis(G, b, diodes, node_index, cont=True)
        else:
            print('Running linear DC analysis')
            x = lin_dc_analysis(G, b)
        printx(x, node_index, cur_index)
    elif dir_type == '.TRAN': 
        print('Running Transient analysis')
        xfinal, full_x, stepsize = tran_analysis(G, C, b, diodes, node_index, cur_index, insources, parts)
        plot(full_x, node_index,stepsize)
        x = np.zeros(len(G),dtype=float)
    else:
        x = np.zeros(len(G),dtype=float)
        return G, C, b, node_index, cur_index, x
    
    return G, C, b, node_index, cur_index, x
    
init_guess = False
G, C, b, node_index, cur_index, x = run_netlist(show_matrices=False)
# G, C, b, node_index, cur_index, x = run_netlist()

In [ ]:
import shutil
import subprocess
import tempfile

NGTEST = True
# Change this if you keep ngspice in a different location.
local_ngspice = Path('ngspice-47_64') / 'Spice64' / 'bin' / 'ngspice.exe'
NGSPICE = str(local_ngspice.resolve()) if local_ngspice.is_file() else shutil.which('ngspice')
TEST_FOLDER = Path('test_cases')
COMPARE_PLOTS = True

if NGTEST:
    if not NGSPICE:
        raise FileNotFoundError(
            
            'ngspice not found. Set NGSPICE to the executable path, '
            'or set NGTEST=False to use the solver without comparisons.'
        )
    print('Using:', NGSPICE)

In [ ]:
# DC and transient reference generation. No compare_spice.py dependency.
def make_ngspice_reference(comps, node_index, cur_index,
                           parts=None, refinement=10):
    """Translate custom netlists; transient reference starts with zero energy."""
    parts = ['.DC'] if parts is None else parts
    if isinstance(parts, str):
        parts = parts.split()
    analysis = parts[0].upper()
    if analysis not in ('.DC', '.TRAN'):
        raise ValueError('Comparisons support .DC and .TRAN only')
    if not isinstance(refinement, int) or refinement < 1:
        raise ValueError('refinement must be a positive integer')

    prefixes = {'R': 'R', 'C': 'C', 'L': 'L', 'V': 'V', 'I': 'I',
                'ISC': 'V', 'VCVS': 'E', 'VCCS': 'G',
                'CCVS': 'H', 'CCCS': 'F', 'D': 'B'}
    spice_names = {c[0]: f'{prefixes[get_type(c[0])]}REF{k}'
                   for k, c in enumerate(comps)}
    spice_nodes = {n: ('0' if index is None else f'NREF{k}')
                   for k, (n, index) in enumerate(node_index.items())}
    lines = ['Reference circuit']

    for comp in comps:
        name = comp[0]
        kind = get_type(name)
        sn = spice_names[name]
        plus, minus = (spice_nodes[n] for n in comp[1:3])

        if kind == 'ISC':
            lines.append(f'{sn} {plus} {minus} DC 0')
        elif kind in ('V', 'I'):
            mode = comp[3]
            amplitude = numeric_value(comp[4], name)
            if mode == 'DC':
                value = f'DC {amplitude:.17g}'
            elif mode in ('SIN', 'COS'):
                if analysis == '.DC':
                    raise ValueError(f'{name}: use DC sources for .DC tests')
                frequency = numeric_value(comp[5], name)
                phase = 0 if mode == 'SIN' else 90
                value = f'SIN(0 {amplitude:.17g} {frequency:.17g} 0 0 {phase})'
            else:
                raise ValueError(f'{name}: unsupported source type {mode}')
            lines.append(f'{sn} {plus} {minus} {value}')
        elif kind in ('R', 'C', 'L'):
            ic = ' IC=0' if analysis == '.TRAN' and kind in ('C', 'L') else ''
            lines.append(f'{sn} {plus} {minus} {comp[3]}{ic}')
        elif kind in ('VCVS', 'VCCS'):
            cp, cm = (spice_nodes[n] for n in comp[3:5])
            lines.append(f'{sn} {plus} {minus} {cp} {cm} {comp[5]}')
        elif kind in ('CCVS', 'CCCS'):
            sense = spice_names[comp[3]]
            lines.append(f'{sn} {plus} {minus} {sense} {comp[4]}')
        elif kind == 'D':
            # Exact match to fill_diodes(Is=1e-15, Vt=0.025).
            lines.append(f'{sn} {plus} {minus} '
                         f'I=1e-15*(exp(V({plus},{minus})/0.025)-1)')

    expressions = {n: f'v({spice_nodes[n].lower()})'
                   for n, index in node_index.items() if index is not None}
    inductor_vectors = []
    for comp in comps:
        label = 'I' + comp[0]
        if label not in cur_index:
            continue
        sn = spice_names[comp[0]].lower()
        if get_type(comp[0]) == 'L':
            expr = f'@{sn}[i]'
            inductor_vectors.append(expr)
        else:
            expr = f'{sn}#branch'
        expressions[label] = expr

    lines.append('.save all ' + ' '.join(inductor_vectors))
    lines += ['.options reltol=1e-8 abstol=1e-13 vntol=1e-10',
              '.options method=trap maxord=1',
              '.control', 'set numdgt=17', 'set wr_singlescale',
              'set wr_vecnames']
    if analysis == '.DC':
        lines.append('op')
    else:
        duration = parse_time(parts[1])
        h = parse_time(parts[2]) if len(parts) == 3 else duration / 100
        href = min(h / refinement, duration)
        # UIC skips the initial operating point; C/L have IC=0 above.
        lines.append(f'tran {href:.17g} {duration:.17g} 0 {href:.17g} uic')

    measurements = []
    for k, (label, expression) in enumerate(expressions.items()):
        alias = f'check_{k}'
        lines.append(f'let {alias} = {expression}')
        measurements.append((label, alias))
    aliases = ' '.join(alias for _, alias in measurements)
    lines += [f'wrdata reference_values.txt {aliases}', 'quit', '.endc', '.end']
    return '\n'.join(lines) + '\n', measurements


def run_ngspice_reference(netlist, measurements, ngspice_command):
    """Read a table containing one scale column and one column per unknown."""
    candidate = Path(ngspice_command).expanduser()
    command = str(candidate.resolve()) if candidate.is_file() else shutil.which(str(ngspice_command))
    if command is None:
        raise FileNotFoundError(f'ngspice executable not found: {ngspice_command}')

    with tempfile.TemporaryDirectory() as folder:
        folder = Path(folder)
        (folder / 'reference.cir').write_text(netlist, encoding='utf-8')
        result = subprocess.run(
            [command, '-b', '-o', 'reference.log', 'reference.cir'],
            cwd=folder, capture_output=True, text=True, timeout=120
        )
        logfile = folder / 'reference.log'
        log = logfile.read_text(errors='replace') if logfile.exists() else ''
        transcript = '\n'.join((log, result.stdout, result.stderr))
        table_path = folder / 'reference_values.txt'
        aborted = 'simulation(s) aborted' in transcript.lower()
        if result.returncode != 0 or aborted or not table_path.is_file():
            raise RuntimeError('ngspice reference failed:\n' + transcript[-6000:])
        header = table_path.read_text().splitlines()[0].split()
        expected = [alias for _, alias in measurements]
        if len(header) != len(expected) + 1 or header[1:] != expected:
            raise RuntimeError('Missing/unexpected ngspice columns:\n' + transcript[-6000:])
        table = np.loadtxt(table_path, skiprows=1, ndmin=2)
        if table.shape[1] != len(expected) + 1 or not np.all(np.isfinite(table)):
            raise RuntimeError('Invalid/nonfinite ngspice reference data')

    return table[:, 0], {
        label: table[:, column + 1]
        for column, (label, _) in enumerate(measurements)
    }

In [ ]:
import contextlib
import io
import math

# DC uses strict numerical tolerances. Transient uses peak-scaled tolerances
# because fixed-step BE differs from a much finer reference integration.
# Change these explicitly if your assignment requires different accuracy.
TRAN_RTOL = 0.03
TRAN_V_ATOL = 1e-3
TRAN_I_ATOL = 1e-6
REFERENCE_REFINEMENT = 10


def compare_case(path, atol=1e-8, rtol=1e-5,
                 tran_rtol=TRAN_RTOL, plot_compare=COMPARE_PLOTS):
    _, comps, directive = read_netlist(path)
    parts = directive.split()
    analysis = parts[0]
    if analysis not in ('.DC', '.TRAN'):
        raise ValueError(f'{analysis} comparison is not implemented')

    # Build once; compare the solver's actual transient history, not a zero
    # vector or only the last solution returned by an older run_netlist().
    G, C, b, nodes, currents = gen_matrices(comps)
    G, C, diodes, insources = fill_matrices(comps, G, C, nodes, currents)
    with contextlib.redirect_stdout(io.StringIO()):
        if analysis == '.DC':
            if any(s[3] != 'DC' for s in insources):
                raise ValueError('.DC comparisons require DC sources')
            b = fill_sources(insources, b, nodes, currents, t=0.0)
            x = (nlin_dc_analysis(G, b, diodes, nodes, cont=True)
                 if diodes else lin_dc_analysis(G, b))
        else:
            xfinal, full_x, stepsize = tran_analysis(
                G, C, b, diodes, nodes, currents, insources, parts
            )
            full_x = np.asarray(full_x, dtype=float)
            duration = parse_time(parts[1])
            times = np.minimum((np.arange(len(full_x)) + 1) * stepsize, duration)
            if full_x.ndim != 2 or full_x.shape[1] != len(b):
                raise ValueError('full_x must contain one row per solved time step')
            if not np.allclose(xfinal, full_x[-1]):
                raise ValueError('xfinal does not match the final row of full_x')

    reference_netlist, measurements = make_ngspice_reference(
        comps, nodes, currents, parts, refinement=REFERENCE_REFINEMENT
    )
    reference_times, reference = run_ngspice_reference(
        reference_netlist, measurements, NGSPICE
    )
    indices = {n: i for n, i in nodes.items() if i is not None}
    indices.update(currents)
    passed = True

    if analysis == '.TRAN':
        # Remove any duplicate timestamps, then interpolate reference data
        # onto the custom solver's h, 2h, ... time points (no t=0 row).
        order = np.argsort(reference_times, kind='stable')
        rt = reference_times[order]
        keep = np.r_[rt[1:] != rt[:-1], True]
        rt = rt[keep]
        slack = 1e-10 * max(duration, 1e-12)
        if rt[0] > times[0] + slack or rt[-1] < times[-1] - slack:
            raise RuntimeError('ngspice reference does not cover the solver times')
        print(f'TRAN: {len(times)} samples; h={stepsize:g} s; '
              f'waveform tolerance={tran_rtol:.1%} of reference peak + absolute')

    for name, index in indices.items():
        if analysis == '.DC':
            actual, expected = float(x[index]), float(reference[name][-1])
            ok = math.isfinite(actual) and math.isclose(
                actual, expected, abs_tol=atol, rel_tol=rtol
            )
            print(f'{name:18} MNA={actual: .9g}  ngspice={expected: .9g}  '
                  f'{"PASS" if ok else "FAIL"}')
        else:
            actual = full_x[:, index]
            expected = np.interp(times, rt, reference[name][order][keep])
            absolute = TRAN_I_ATOL if name in currents else TRAN_V_ATOL
            tolerance = absolute + tran_rtol * np.max(np.abs(expected))
            error = np.abs(actual - expected)
            ok = bool(np.all(np.isfinite(actual)) and np.all(error <= tolerance))
            worst = int(np.argmax(error))
            print(f'{name:18} max error={error[worst]:.6g}  '
                  f'limit={tolerance:.6g}  at t={times[worst]:.6g} s  '
                  f'{"PASS" if ok else "FAIL"}')
            if plot_compare:
                plt.figure()
                plt.plot(times, actual, label='Your Backward Euler')
                plt.plot(times, expected, '--', label='ngspice reference')
                plt.xlabel('Time (s)')
                plt.ylabel(f'{name} ({"A" if name in currents else "V"})')
                plt.legend()
                plt.grid(True)
                plt.show()
        passed = passed and ok
    return passed


# Run this cell after the ngspice setup cell. Setting NGTEST=False skips it.
if NGTEST:
    cases = sorted(TEST_FOLDER.glob('*.cir'))
    if not cases:
        raise FileNotFoundError(f'No .cir files found in {TEST_FOLDER.resolve()}')
    results = []
    for case in cases:
        print(f'\n--- {case.name} ---')
        try:
            results.append(compare_case(case))
        except Exception as error:
            results.append(False)
            print(f'ERROR: {type(error).__name__}: {error}')
    print(f'\n{sum(results)}/{len(results)} test cases passed.')